# Decomposed Teacher Construction

## Data

- Panel: `modeling_panels/model_panel_start_1970_01_31.parquet`
- Sample in this run: 1970-01-31 to 2026-03-31
- Monthly observations: 675

## Selected state blocks

- Equity opportunity: `EO3`
- Equity risk: `ER3`
- Bond opportunity: `BO8`
- Bond rate risk: `BRR1`
- Bond credit risk: `BRC2`

## Selected binary states

- `EO3_3m__maj3_mr3`
- `EO3_6m__maj3_mr3`
- `ER3_3m__maj3_mr3`
- `ER3_6m__maj3_mr3`
- `BO8_3m__maj3_mr3`
- `BO8_6m__maj3_mr3`
- `BRR1_3m__maj3_mr3`
- `BRR1_6m__maj3_mr3`
- `BRC2_3m__maj3_mr3`
- `BRC2_6m__maj3_mr3`

## Continuous score definitions

For horizon $h \in \{3,6\}$:

### Equity opportunity
$$
S^{EO3}_{t,h} = x^{eq}_{t,h} - 0.75\,|MDD^{eq,fwd}_{t,h}|
$$

- $x^{eq}_{t,h}$: forward equity excess return over $t \to t+h$
- $MDD^{eq,fwd}_{t,h}$: forward maximum drawdown from the reconstructed monthly equity path over $t \to t+h$

### Equity risk
$$
S^{ER3}_{t,h} = 0.4\,z(\overline{RV60}_{t,h}) + 0.3\,z(\overline{VIX}_{t,h}) + 0.3\,z(|MDD^{eq,fwd}_{t,h}|)
$$

- $\overline{RV60}_{t,h}$: forward-window average of `rv_60d`
- $\overline{VIX}_{t,h}$: forward-window average of `VIXCLSx`

### Bond opportunity
$$
S^{BO8}_{t,h} = x^{fi}_{t,h} + 0.5\,z(\overline{slope}_{t,h}) - 0.5\,z(|\Delta y^{fwd}_{t,h}|) - 0.5\,z(\overline{\pi}_{t,h})
$$

- $x^{fi}_{t,h}$: forward bond excess return over $t \to t+h$
- $\overline{slope}_{t,h}$: forward-window average of `derived_slope_10y_3m_wrds`
- $|\Delta y^{fwd}_{t,h}|$: absolute change in 10Y yield from $t$ to $t+h$
- $\overline{\pi}_{t,h}$: forward-window average of core CPI

### Bond rate risk
$$
S^{BRR1}_{t,h} = z(|\Delta y^{fwd}_{t,h}|)
$$

### Bond credit risk
$$
S^{BRC2}_{t,h} = z(\Delta cs^{fwd}_{t,h})
$$

- $\Delta cs^{fwd}_{t,h}$: credit-spread change from $t$ to $t+h$

## Binary-state engine

For each selected continuous score $S_{t,h}$:

1. Full-sample median threshold:
$$
\tilde R_{t,h} = \mathbf{1}\{S_{t,h} > \operatorname{median}(S_{\cdot,h})\}
$$
2. Backward 3-month majority vote
3. Minimum-run enforcement with min-run $=3$


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 220)

PANEL_PATH = Path("modeling_panels/model_panel_start_1970_01_31.parquet")

panel = pd.read_parquet(PANEL_PATH).copy()
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values("date").reset_index(drop=True)

print(f"Loaded: {PANEL_PATH}")
print(f"Shape: {panel.shape}")
print(f"Date range: {panel['date'].min().date()} -> {panel['date'].max().date()}")

## Working variables

In [ ]:
alias_map = {
    "eq_fwd_excess_3m": "sp500_fwd_excess_3m",
    "eq_fwd_excess_6m": "sp500_fwd_excess_6m",
    "eq_ret_1m": "sprtrn_sp500",
    "eq_vol_60d": "rv_60d",
    "eq_vix": "VIXCLSx",
    "fi_fwd_excess_3m": "bond_fwd_excess_3m",
    "fi_fwd_excess_6m": "bond_fwd_excess_6m",
    "fi_ret_1m": "agg_ret",
    "y10": "GS10",
    "curve_10y3m": "derived_slope_10y_3m_wrds",
    "curve_curv": "derived_curve_curvature_wrds",
    "credit_spread": "credit_spread_baa_aaa",
    "core_cpi": "CPILFESL",
}

df = pd.DataFrame({"date": panel["date"]}).copy()
for new_name, old_name in alias_map.items():
    df[new_name] = pd.to_numeric(panel[old_name], errors="coerce")

print(df.columns.to_list())

## Forward-window realized objects

In [ ]:
def forward_window_mean(x: pd.Series, horizon: int) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    out = pd.Series(index=x.index, dtype=float)
    for i in range(len(x)):
        win = x.iloc[i + 1:i + 1 + horizon]
        out.iloc[i] = win.mean() if len(win) == horizon and not win.isna().any() else np.nan
    return out

def forward_change(x: pd.Series, horizon: int) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    out = pd.Series(index=x.index, dtype=float)
    for i in range(len(x)):
        j = i + horizon
        if j < len(x) and pd.notna(x.iloc[i]) and pd.notna(x.iloc[j]):
            out.iloc[i] = x.iloc[j] - x.iloc[i]
        else:
            out.iloc[i] = np.nan
    return out

def forward_abs_change(x: pd.Series, horizon: int) -> pd.Series:
    return forward_change(x, horizon).abs()

def future_realized_vol(ret: pd.Series, horizon: int) -> pd.Series:
    ret = pd.to_numeric(ret, errors="coerce")
    out = pd.Series(index=ret.index, dtype=float)
    for i in range(len(ret)):
        win = ret.iloc[i + 1:i + 1 + horizon]
        out.iloc[i] = win.std(ddof=0) if len(win) == horizon and not win.isna().any() else np.nan
    return out

def future_downside_frequency(ret: pd.Series, horizon: int) -> pd.Series:
    ret = pd.to_numeric(ret, errors="coerce")
    out = pd.Series(index=ret.index, dtype=float)
    for i in range(len(ret)):
        win = ret.iloc[i + 1:i + 1 + horizon]
        out.iloc[i] = float((win < 0).mean()) if len(win) == horizon and not win.isna().any() else np.nan
    return out

def forward_max_drawdown_from_returns(ret: pd.Series, horizon: int, start_value: float = 100.0) -> pd.Series:
    ret = pd.to_numeric(ret, errors="coerce")
    out = pd.Series(index=ret.index, dtype=float)
    for i in range(len(ret)):
        win = ret.iloc[i + 1:i + 1 + horizon]
        if len(win) < horizon or win.isna().any():
            out.iloc[i] = np.nan
            continue
        levels = [start_value]
        level = start_value
        for r in win:
            level *= (1.0 + r)
            levels.append(level)
        levels = np.array(levels, dtype=float)
        running_max = np.maximum.accumulate(levels)
        dd = levels / running_max - 1.0
        out.iloc[i] = dd.min()
    return out

def oracle_expanding_zscore(x: pd.Series, min_periods: int = 24) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    mu = x.expanding(min_periods=min_periods).mean()
    sd = x.expanding(min_periods=min_periods).std(ddof=0)
    return (x - mu) / sd.replace(0.0, np.nan)

for h in [3, 6]:
    df[f"eq_fwd_mdd_{h}m"] = forward_max_drawdown_from_returns(df["eq_ret_1m"], h)
    df[f"eq_fwd_realized_vol_{h}m"] = future_realized_vol(df["eq_ret_1m"], h)
    df[f"eq_fwd_downside_freq_{h}m"] = future_downside_frequency(df["eq_ret_1m"], h)
    df[f"eq_fwd_rv60_mean_{h}m"] = forward_window_mean(df["eq_vol_60d"], h)
    df[f"eq_fwd_vix_mean_{h}m"] = forward_window_mean(df["eq_vix"], h)

    df[f"fi_fwd_realized_vol_{h}m"] = future_realized_vol(df["fi_ret_1m"], h)
    df[f"fi_fwd_downside_freq_{h}m"] = future_downside_frequency(df["fi_ret_1m"], h)
    df[f"curve_10y3m_fwd_mean_{h}m"] = forward_window_mean(df["curve_10y3m"], h)
    df[f"dy10_abs_fwd_{h}m"] = forward_abs_change(df["y10"], h)
    df[f"core_cpi_fwd_mean_{h}m"] = forward_window_mean(df["core_cpi"], h)
    df[f"credit_spread_change_fwd_{h}m"] = forward_change(df["credit_spread"], h)

print([
    "eq_fwd_mdd_3m", "eq_fwd_mdd_6m", "eq_fwd_rv60_mean_3m", "eq_fwd_rv60_mean_6m",
    "eq_fwd_vix_mean_3m", "eq_fwd_vix_mean_6m", "curve_10y3m_fwd_mean_3m", "curve_10y3m_fwd_mean_6m",
    "dy10_abs_fwd_3m", "dy10_abs_fwd_6m", "credit_spread_change_fwd_3m", "credit_spread_change_fwd_6m"
])

## Selected continuous scores

In [ ]:
score_df = pd.DataFrame({"date": df["date"]}).copy()

for h in [3, 6]:
    df[f"z_eq_fwd_mdd_abs_{h}m"] = oracle_expanding_zscore(df[f"eq_fwd_mdd_{h}m"].abs())
    df[f"z_eq_fwd_rv60_mean_{h}m"] = oracle_expanding_zscore(df[f"eq_fwd_rv60_mean_{h}m"])
    df[f"z_eq_fwd_vix_mean_{h}m"] = oracle_expanding_zscore(df[f"eq_fwd_vix_mean_{h}m"])
    df[f"z_curve_10y3m_fwd_mean_{h}m"] = oracle_expanding_zscore(df[f"curve_10y3m_fwd_mean_{h}m"])
    df[f"z_dy10_abs_fwd_{h}m"] = oracle_expanding_zscore(df[f"dy10_abs_fwd_{h}m"])
    df[f"z_core_cpi_fwd_mean_{h}m"] = oracle_expanding_zscore(df[f"core_cpi_fwd_mean_{h}m"])
    df[f"z_credit_spread_change_fwd_{h}m"] = oracle_expanding_zscore(df[f"credit_spread_change_fwd_{h}m"])

    score_df[f"EO3_{h}m"] = df[f"eq_fwd_excess_{h}m"] - 0.75 * df[f"eq_fwd_mdd_{h}m"].abs()
    score_df[f"ER3_{h}m"] = (
        0.4 * df[f"z_eq_fwd_rv60_mean_{h}m"]
        + 0.3 * df[f"z_eq_fwd_vix_mean_{h}m"]
        + 0.3 * df[f"z_eq_fwd_mdd_abs_{h}m"]
    )
    score_df[f"BO8_{h}m"] = (
        df[f"fi_fwd_excess_{h}m"]
        + 0.5 * df[f"z_curve_10y3m_fwd_mean_{h}m"]
        - 0.5 * df[f"z_dy10_abs_fwd_{h}m"]
        - 0.5 * df[f"z_core_cpi_fwd_mean_{h}m"]
    )
    score_df[f"BRR1_{h}m"] = df[f"z_dy10_abs_fwd_{h}m"]
    score_df[f"BRC2_{h}m"] = df[f"z_credit_spread_change_fwd_{h}m"]

print(score_df.columns.to_list())

## Binary state construction

In [ ]:
def full_sample_median_threshold(score: pd.Series) -> pd.Series:
    score = pd.to_numeric(score, errors="coerce")
    out = pd.Series(index=score.index, dtype=float)
    valid = score.notna()
    if valid.sum() == 0:
        out[:] = np.nan
        return out
    med = score.loc[valid].median()
    out.loc[valid] = (score.loc[valid] > med).astype(float)
    out.loc[~valid] = np.nan
    return out

def backward_majority_vote(binary: pd.Series, window: int = 3) -> pd.Series:
    x = pd.to_numeric(binary, errors="coerce")
    out = pd.Series(index=x.index, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        win = x.iloc[lo:i + 1].dropna()
        out.iloc[i] = (1.0 if win.mean() >= 0.5 else 0.0) if len(win) > 0 else np.nan
    return out

def enforce_min_run(binary: pd.Series, min_run: int = 3) -> pd.Series:
    x = pd.to_numeric(binary, errors="coerce")
    arr = x.to_numpy(copy=True, dtype=float)
    n = len(arr)
    i = 0
    while i < n:
        if np.isnan(arr[i]):
            i += 1
            continue
        j = i + 1
        while j < n and not np.isnan(arr[j]) and arr[j] == arr[i]:
            j += 1
        run_len = j - i
        if run_len < min_run:
            prev_val = np.nan
            k = i - 1
            while k >= 0:
                if not np.isnan(arr[k]):
                    prev_val = arr[k]
                    break
                k -= 1
            next_val = np.nan
            k = j
            while k < n:
                if not np.isnan(arr[k]):
                    next_val = arr[k]
                    break
                k += 1
            fill = prev_val if not np.isnan(prev_val) else next_val
            if not np.isnan(fill):
                arr[i:j] = fill
        i = j
    return pd.Series(arr, index=x.index)

teacher_state_df = pd.DataFrame({"date": score_df["date"]}).copy()

for c in [x for x in score_df.columns if x != "date"]:
    raw = full_sample_median_threshold(score_df[c])
    maj3 = backward_majority_vote(raw, window=3)
    mr3 = enforce_min_run(maj3, min_run=3)
    teacher_state_df[f"{c}__maj3_mr3"] = mr3

selected_state_cols = [c for c in teacher_state_df.columns if c != "date"]
print(selected_state_cols)

## State diagnostics

In [ ]:
diag = df[["date"]].copy()
for h in [3, 6]:
    diag[f"eq_fwd_excess_{h}m"] = df[f"eq_fwd_excess_{h}m"]
    diag[f"fi_fwd_excess_{h}m"] = df[f"fi_fwd_excess_{h}m"]
    diag[f"eq_fwd_realized_vol_{h}m"] = df[f"eq_fwd_realized_vol_{h}m"]
    diag[f"fi_fwd_realized_vol_{h}m"] = df[f"fi_fwd_realized_vol_{h}m"]
    diag[f"eq_fwd_downside_freq_{h}m"] = df[f"eq_fwd_downside_freq_{h}m"]
    diag[f"fi_fwd_downside_freq_{h}m"] = df[f"fi_fwd_downside_freq_{h}m"]
    diag[f"abs_dy10_fwd_{h}m"] = df[f"dy10_abs_fwd_{h}m"]
    diag[f"spread_widen_fwd_{h}m"] = df[f"credit_spread_change_fwd_{h}m"]

def run_lengths(x: pd.Series):
    vals = x.dropna().astype(int).to_numpy()
    if len(vals) == 0:
        return []
    runs = []
    curr = vals[0]
    length = 1
    for v in vals[1:]:
        if v == curr:
            length += 1
        else:
            runs.append(length)
            curr = v
            length = 1
    runs.append(length)
    return runs

def binary_summary_metrics(state: pd.Series):
    s = state.dropna().astype(int)
    lag = s.shift(1)
    switches = (s != lag).dropna()
    runs = run_lengths(s)
    return {
        "n_obs": int(len(s)),
        "class_1_share": float(s.mean()),
        "switch_rate": float(switches.mean()) if len(switches) > 0 else np.nan,
        "avg_run_length": float(np.mean(runs)) if len(runs) > 0 else np.nan,
        "median_run_length": float(np.median(runs)) if len(runs) > 0 else np.nan,
    }

def conditional_state_mean(state: pd.Series, y: pd.Series, state_value: int):
    tmp = pd.DataFrame({"s": state, "y": y}).dropna()
    return float(tmp.groupby("s")["y"].mean().get(state_value, np.nan))

rows = []
for state_col in selected_state_cols:
    base = state_col.split("__")[0]
    horizon = "3m" if "_3m" in base else "6m"
    block = (
        "equity_opportunity" if base.startswith("EO") else
        "equity_risk" if base.startswith("ER") else
        "bond_opportunity" if base.startswith("BO") else
        "bond_rate_risk" if base.startswith("BRR") else
        "bond_credit_risk"
    )
    state = teacher_state_df[state_col]
    row = {"teacher_col": state_col, "block": block, "horizon": horizon}
    row.update(binary_summary_metrics(state))

    if block == "equity_opportunity":
        row["mean_state0"] = conditional_state_mean(state, diag[f"eq_fwd_excess_{horizon}"], 0)
        row["mean_state1"] = conditional_state_mean(state, diag[f"eq_fwd_excess_{horizon}"], 1)
        row["future_vol_state0"] = conditional_state_mean(state, diag[f"eq_fwd_realized_vol_{horizon}"], 0)
        row["future_vol_state1"] = conditional_state_mean(state, diag[f"eq_fwd_realized_vol_{horizon}"], 1)
        row["future_downside_state0"] = conditional_state_mean(state, diag[f"eq_fwd_downside_freq_{horizon}"], 0)
        row["future_downside_state1"] = conditional_state_mean(state, diag[f"eq_fwd_downside_freq_{horizon}"], 1)

    elif block == "equity_risk":
        row["mean_state0"] = conditional_state_mean(state, diag[f"eq_fwd_excess_{horizon}"], 0)
        row["mean_state1"] = conditional_state_mean(state, diag[f"eq_fwd_excess_{horizon}"], 1)
        row["future_vol_state0"] = conditional_state_mean(state, diag[f"eq_fwd_realized_vol_{horizon}"], 0)
        row["future_vol_state1"] = conditional_state_mean(state, diag[f"eq_fwd_realized_vol_{horizon}"], 1)
        row["future_downside_state0"] = conditional_state_mean(state, diag[f"eq_fwd_downside_freq_{horizon}"], 0)
        row["future_downside_state1"] = conditional_state_mean(state, diag[f"eq_fwd_downside_freq_{horizon}"], 1)

    elif block == "bond_opportunity":
        row["mean_state0"] = conditional_state_mean(state, diag[f"fi_fwd_excess_{horizon}"], 0)
        row["mean_state1"] = conditional_state_mean(state, diag[f"fi_fwd_excess_{horizon}"], 1)
        row["future_vol_state0"] = conditional_state_mean(state, diag[f"fi_fwd_realized_vol_{horizon}"], 0)
        row["future_vol_state1"] = conditional_state_mean(state, diag[f"fi_fwd_realized_vol_{horizon}"], 1)
        row["future_downside_state0"] = conditional_state_mean(state, diag[f"fi_fwd_downside_freq_{horizon}"], 0)
        row["future_downside_state1"] = conditional_state_mean(state, diag[f"fi_fwd_downside_freq_{horizon}"], 1)

    elif block == "bond_rate_risk":
        row["mean_state0"] = conditional_state_mean(state, diag[f"fi_fwd_excess_{horizon}"], 0)
        row["mean_state1"] = conditional_state_mean(state, diag[f"fi_fwd_excess_{horizon}"], 1)
        row["future_vol_state0"] = conditional_state_mean(state, diag[f"fi_fwd_realized_vol_{horizon}"], 0)
        row["future_vol_state1"] = conditional_state_mean(state, diag[f"fi_fwd_realized_vol_{horizon}"], 1)
        row["future_abs_dy10_state0"] = conditional_state_mean(state, diag[f"abs_dy10_fwd_{horizon}"], 0)
        row["future_abs_dy10_state1"] = conditional_state_mean(state, diag[f"abs_dy10_fwd_{horizon}"], 1)

    else:
        row["mean_state0"] = conditional_state_mean(state, diag[f"fi_fwd_excess_{horizon}"], 0)
        row["mean_state1"] = conditional_state_mean(state, diag[f"fi_fwd_excess_{horizon}"], 1)
        row["future_vol_state0"] = conditional_state_mean(state, diag[f"fi_fwd_realized_vol_{horizon}"], 0)
        row["future_vol_state1"] = conditional_state_mean(state, diag[f"fi_fwd_realized_vol_{horizon}"], 1)
        row["future_spread_widen_state0"] = conditional_state_mean(state, diag[f"spread_widen_fwd_{horizon}"], 0)
        row["future_spread_widen_state1"] = conditional_state_mean(state, diag[f"spread_widen_fwd_{horizon}"], 1)

    rows.append(row)

selected_diag_df = pd.DataFrame(rows)
selected_diag_df["gap_1_minus_0"] = selected_diag_df["mean_state1"] - selected_diag_df["mean_state0"]
print(selected_diag_df.to_string(index=False))

## Within-asset dependence

In [ ]:
survivor_candidates = {
    "eq_opp_3m": "EO3_3m__maj3_mr3",
    "eq_opp_6m": "EO3_6m__maj3_mr3",
    "eq_risk_3m": "ER3_3m__maj3_mr3",
    "eq_risk_6m": "ER3_6m__maj3_mr3",
    "fi_opp_3m": "BO8_3m__maj3_mr3",
    "fi_opp_6m": "BO8_6m__maj3_mr3",
    "fi_rate_risk_3m": "BRR1_3m__maj3_mr3",
    "fi_rate_risk_6m": "BRR1_6m__maj3_mr3",
    "fi_credit_risk_3m": "BRC2_3m__maj3_mr3",
    "fi_credit_risk_6m": "BRC2_6m__maj3_mr3",
}

pairs = [
    ("eq_opp_3m", "eq_risk_3m"),
    ("eq_opp_6m", "eq_risk_6m"),
    ("fi_opp_3m", "fi_rate_risk_3m"),
    ("fi_opp_6m", "fi_rate_risk_6m"),
    ("fi_opp_3m", "fi_credit_risk_3m"),
    ("fi_opp_6m", "fi_credit_risk_6m"),
]

for a_key, b_key in pairs:
    a = survivor_candidates[a_key]
    b = survivor_candidates[b_key]
    tmp = pd.DataFrame({"a": teacher_state_df[a], "b": teacher_state_df[b]}).dropna()
    corr = float(tmp[["a", "b"]].corr().iloc[0, 1])
    joint = pd.crosstab(tmp["a"], tmp["b"], normalize="all")
    print("\n" + "=" * 100)
    print(f"{a_key} vs {b_key}")
    print(f"A = {a}")
    print(f"B = {b}")
    print(f"corr = {corr:.6f}")
    print(joint.to_string())

## Cross-asset dependence

In [ ]:
cross_state_cols = {
    "eq_opp_3m": "EO3_3m__maj3_mr3",
    "eq_opp_6m": "EO3_6m__maj3_mr3",
    "eq_risk_3m": "ER3_3m__maj3_mr3",
    "eq_risk_6m": "ER3_6m__maj3_mr3",
    "fi_opp_3m": "BO8_3m__maj3_mr3",
    "fi_opp_6m": "BO8_6m__maj3_mr3",
    "fi_rate_risk_3m": "BRR1_3m__maj3_mr3",
    "fi_rate_risk_6m": "BRR1_6m__maj3_mr3",
    "fi_credit_risk_3m": "BRC2_3m__maj3_mr3",
    "fi_credit_risk_6m": "BRC2_6m__maj3_mr3",
}

state_df_for_matrix = pd.DataFrame({k: teacher_state_df[v] for k, v in cross_state_cols.items()}).dropna()
print(state_df_for_matrix.corr().to_string())

## Final selected state set

In [ ]:
chosen_survivors = {
    "equity_opportunity": ["EO3_3m__maj3_mr3", "EO3_6m__maj3_mr3"],
    "equity_risk": ["ER3_3m__maj3_mr3", "ER3_6m__maj3_mr3"],
    "bond_opportunity": ["BO8_3m__maj3_mr3", "BO8_6m__maj3_mr3"],
    "bond_rate_risk": ["BRR1_3m__maj3_mr3", "BRR1_6m__maj3_mr3"],
    "bond_credit_risk": ["BRC2_3m__maj3_mr3", "BRC2_6m__maj3_mr3"],
}

print(chosen_survivors)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# STATE OVERLAY GRID PLOTS
# - Reconstruct normalized asset levels from monthly returns
# - Shade periods where the selected binary state = 1
# - One subplot per selected state
# ============================================================

# ---------- helpers ----------
def build_level_from_returns(ret: pd.Series, start_value: float = 100.0) -> pd.Series:
    r = pd.to_numeric(ret, errors="coerce").fillna(0.0)
    return start_value * (1.0 + r).cumprod()

def contiguous_true_segments(mask: pd.Series):
    """
    Return list of (start_idx, end_idx) for contiguous True segments.
    end_idx is inclusive.
    """
    x = mask.fillna(False).astype(bool).to_numpy()
    segments = []
    start = None
    for i, val in enumerate(x):
        if val and start is None:
            start = i
        if (not val) and start is not None:
            segments.append((start, i - 1))
            start = None
    if start is not None:
        segments.append((start, len(x) - 1))
    return segments

def shade_state_regions(ax, dates: pd.Series, state: pd.Series, alpha: float = 0.22):
    mask = state.eq(1.0)
    segs = contiguous_true_segments(mask)
    for s, e in segs:
        x0 = dates.iloc[s]
        x1 = dates.iloc[e]
        ax.axvspan(x0, x1, alpha=alpha)

# ---------- reconstruct normalized levels ----------
plot_df = pd.DataFrame({"date": df["date"]}).copy()
plot_df["sp500_level"] = build_level_from_returns(df["eq_ret_1m"], start_value=100.0)
plot_df["agg_level"] = build_level_from_returns(df["fi_ret_1m"], start_value=100.0)

# ---------- selected states and metadata ----------
plot_specs = [
    {
        "state_col": "EO3_3m__maj3_mr3",
        "asset_col": "sp500_level",
        "asset_label": "S&P 500 level (normalized to 100)",
        "title": "EO3 3M | Equity opportunity",
    },
    {
        "state_col": "EO3_6m__maj3_mr3",
        "asset_col": "sp500_level",
        "asset_label": "S&P 500 level (normalized to 100)",
        "title": "EO3 6M | Equity opportunity",
    },
    {
        "state_col": "ER3_3m__maj3_mr3",
        "asset_col": "sp500_level",
        "asset_label": "S&P 500 level (normalized to 100)",
        "title": "ER3 3M | Equity risk",
    },
    {
        "state_col": "ER3_6m__maj3_mr3",
        "asset_col": "sp500_level",
        "asset_label": "S&P 500 level (normalized to 100)",
        "title": "ER3 6M | Equity risk",
    },
    {
        "state_col": "BO8_3m__maj3_mr3",
        "asset_col": "agg_level",
        "asset_label": "Agg bond level (normalized to 100)",
        "title": "BO8 3M | Bond opportunity",
    },
    {
        "state_col": "BO8_6m__maj3_mr3",
        "asset_col": "agg_level",
        "asset_label": "Agg bond level (normalized to 100)",
        "title": "BO8 6M | Bond opportunity",
    },
    {
        "state_col": "BRR1_3m__maj3_mr3",
        "asset_col": "agg_level",
        "asset_label": "Agg bond level (normalized to 100)",
        "title": "BRR1 3M | Bond rate risk",
    },
    {
        "state_col": "BRR1_6m__maj3_mr3",
        "asset_col": "agg_level",
        "asset_label": "Agg bond level (normalized to 100)",
        "title": "BRR1 6M | Bond rate risk",
    },
    {
        "state_col": "BRC2_3m__maj3_mr3",
        "asset_col": "agg_level",
        "asset_label": "Agg bond level (normalized to 100)",
        "title": "BRC2 3M | Bond credit risk",
    },
    {
        "state_col": "BRC2_6m__maj3_mr3",
        "asset_col": "agg_level",
        "asset_label": "Agg bond level (normalized to 100)",
        "title": "BRC2 6M | Bond credit risk",
    },
]

# ---------- plot ----------
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(20, 24), sharex=True)
axes = axes.flatten()

for ax, spec in zip(axes, plot_specs):
    state_col = spec["state_col"]
    asset_col = spec["asset_col"]

    tmp = pd.DataFrame({
        "date": plot_df["date"],
        "level": plot_df[asset_col],
        "state": teacher_state_df[state_col]
    }).dropna(subset=["date", "level"])

    # line
    ax.plot(tmp["date"], tmp["level"], linewidth=1.4)

    # shaded state=1 periods
    shade_state_regions(ax, tmp["date"], tmp["state"], alpha=0.22)

    # cosmetics
    share_1 = tmp["state"].dropna().mean()
    ax.set_title(f"{spec['title']}  |  shaded = state 1  |  share(state=1) = {share_1:.3f}", fontsize=11)
    ax.set_ylabel(spec["asset_label"])
    ax.grid(True, alpha=0.25)

# x labels only on bottom row
for ax in axes[-2:]:
    ax.set_xlabel("Date")

plt.suptitle(
    "Selected teacher states over normalized asset levels\nShaded regions indicate state = 1",
    fontsize=16,
    y=0.995
)
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()

## Main findings

- `EO3_3m__maj3_mr3`: state 1 has mean forward equity excess return `0.052711` vs `-0.028590` in state 0; gap `0.081301`. Forward realized vol is lower in state 1 (`0.024506` vs `0.036149`) and downside frequency is lower (`0.249231` vs `0.532934`).
- `EO3_6m__maj3_mr3`: state 1 has mean forward equity excess return `0.097419` vs `-0.046078` in state 0; gap `0.143497`. Forward realized vol is lower in state 1 (`0.028799` vs `0.043474`) and downside frequency is lower (`0.262721` vs `0.514925`).
- `ER3_3m__maj3_mr3`: state 1 has lower forward equity excess return (`-0.002030` vs `0.024717`), higher forward realized vol (`0.036797` vs `0.022508`), and higher downside frequency (`0.469285` vs `0.338574`).
- `ER3_6m__maj3_mr3`: state 1 has lower forward equity excess return (`0.000536` vs `0.044358`), higher forward realized vol (`0.045044` vs `0.026500`), and higher downside frequency (`0.473438` vs `0.337423`).
- `BO8_3m__maj3_mr3`: state 1 has mean forward bond excess return `0.009483` vs `0.001858`; gap `0.007625`. Forward realized vol is lower in state 1 (`0.007920` vs `0.010923`) and downside frequency is lower (`0.305011` vs `0.370748`).
- `BO8_6m__maj3_mr3`: state 1 has mean forward bond excess return `0.021265` vs `0.001645`; gap `0.019620`. Forward realized vol is lower in state 1 (`0.009714` vs `0.013573`) and downside frequency is lower (`0.295380` vs `0.380952`).
- `BRR1_3m__maj3_mr3`: state 1 has larger forward absolute 10Y yield move (`0.602094` vs `0.221032`) and higher forward bond realized vol (`0.012782` vs `0.008205`).
- `BRR1_6m__maj3_mr3`: state 1 has larger forward absolute 10Y yield move (`0.944387` vs `0.311756`) and higher forward bond realized vol (`0.014483` vs `0.011705`).
- `BRC2_3m__maj3_mr3`: state 1 has positive forward spread change (`0.101379`) while state 0 has negative forward spread change (`-0.122226`).
- `BRC2_6m__maj3_mr3`: state 1 has positive forward spread change (`0.197370`) while state 0 has negative forward spread change (`-0.187485`).
- Within equity, opportunity and risk remain distinct: correlation is `-0.317624` at 3M and `-0.324699` at 6M.
- Within fixed income, bond opportunity vs rate risk correlation is `-0.207315` at 3M and `-0.416873` at 6M; bond opportunity vs credit risk correlation is `-0.148360` at 3M and `-0.115290` at 6M.
- Cross-horizon persistence is high within the same block: `eq_risk_3m` vs `eq_risk_6m` correlation `0.839352`, `fi_opp_3m` vs `fi_opp_6m` correlation `0.852518`, `fi_credit_risk_3m` vs `fi_credit_risk_6m` correlation `0.592861`, `eq_opp_3m` vs `eq_opp_6m` correlation `0.650758`.

## Continuation: joint-state construction

Define horizon-specific joint teachers.

### Equity joint states
$$
J^{eq}_{t,3m} = \big(EO3\_3m\_\_maj3\_mr3,\ ER3\_3m\_\_maj3\_mr3\big)
$$

$$
J^{eq}_{t,6m} = \big(EO3\_6m\_\_maj3\_mr3,\ ER3\_6m\_\_maj3\_mr3\big)
$$

Each equity side has 4 raw joint states:

- `00`: weak opportunity, low risk
- `01`: weak opportunity, high risk
- `10`: strong opportunity, low risk
- `11`: strong opportunity, high risk

### Fixed-income joint states
$$
J^{fi}_{t,3m} = \big(BO8\_3m\_\_maj3\_mr3,\ BRR1\_3m\_\_maj3\_mr3,\ BRC2\_3m\_\_maj3\_mr3\big)
$$

$$
J^{fi}_{t,6m} = \big(BO8\_6m\_\_maj3\_mr3,\ BRR1\_6m\_\_maj3\_mr3,\ BRC2\_6m\_\_maj3\_mr3\big)
$$

Each fixed-income side has 8 raw joint states:

- `000`: weak opportunity, low rate risk, low credit risk
- `001`: weak opportunity, low rate risk, high credit risk
- `010`: weak opportunity, high rate risk, low credit risk
- `011`: weak opportunity, high rate risk, high credit risk
- `100`: strong opportunity, low rate risk, low credit risk
- `101`: strong opportunity, low rate risk, high credit risk
- `110`: strong opportunity, high rate risk, low credit risk
- `111`: strong opportunity, high rate risk, high credit risk
